In [1]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
!pip install pyarrow==8.0.0  # Downgrade pyarrow to a known compatible version
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from datasets import load_dataset, Dataset
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 26.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 16.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 19.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 48.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 16.1.0 which

In [34]:
# Step 2: Define Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name_or_path = 'google/muril-base-cased'
tokenizer_name_or_path = 'google/muril-base-cased'
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128
lr = 5e-5  # Adjusted learning rate
num_epochs = 50  # Reduced number of epochs for initial testing
batch_size = 8

# Step 3: Load and Prepare Dataset
column_names = [label_column, "blank", text_column]
dataset = pd.read_csv("/content/drive/MyDrive/societal_2l_train_bn.csv", header=None, names=column_names).head(1000)
dataset.drop("blank", axis=1, inplace=True)
dataset[label_column] = dataset[label_column].map({0: 0, 4: 1})  # Adjust according to your labels

# Check for data balance
print(dataset[label_column].value_counts())

from sklearn.model_selection import train_test_split
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
dataset_train = Dataset.from_pandas(train_dataset)
dataset_test = Dataset.from_pandas(test_dataset)

# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Step 5: Preprocess Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], truncation=True, max_length=max_length, padding="max_length")
    model_inputs["labels"] = examples[label_column]
    return model_inputs

# Step 6: Apply Preprocessing
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

# Step 7: DataLoaders
train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=False, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
print(dataset[label_column].value_counts())


# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)
model = get_peft_model(model, peft_config)
model.dropout = torch.nn.Dropout(0.1)  # Add dropout to prevent overfitting
print(model.print_trainable_parameters())

# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)  # Added weight decay
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))


text_label
0    513
1    487
Name: count, dtype: int64


Running tokenizer on dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

text_label
0    513
1    487
Name: count, dtype: int64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 7,682 || all params: 237,565,444 || trainable%: 0.0032
None


In [35]:
# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    true_labels = []
    for step, batch in enumerate(tqdm(test_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())
        true_labels.extend(batch["labels"].detach().cpu().numpy())

    eval_epoch_loss = eval_loss / len(test_dataloader)
    train_epoch_loss = total_loss / len(train_dataloader)
    accuracy = accuracy_score(true_labels, eval_preds)
    f1 = f1_score(true_labels, eval_preds, average='weighted')
    recall = recall_score(true_labels, eval_preds, average='weighted')
    precision = precision_score(true_labels, eval_preds, average='weighted')

    print(f"Epoch {epoch}: train_loss={train_epoch_loss}, eval_loss={eval_epoch_loss}, accuracy={accuracy}, f1_score={f1}, recall={recall}, precision={precision}")


100%|██████████| 25/25 [00:01<00:00, 14.59it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 0: train_loss=0.69310063123703, eval_loss=0.6932629346847534, accuracy=0.45, f1_score=0.2793103448275862, recall=0.45, precision=0.2025


100%|██████████| 25/25 [00:01<00:00, 14.25it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 1: train_loss=0.6932533979415894, eval_loss=0.6930602788925171, accuracy=0.55, f1_score=0.3903225806451613, recall=0.55, precision=0.30250000000000005


100%|██████████| 25/25 [00:01<00:00, 14.64it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

100%|██████████| 25/25 [00:01<00:00, 14.47it/s]
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 48: train_loss=0.6907498240470886, eval_loss=0.689780056476593, accuracy=0.55, f1_score=0.3903225806451613, recall=0.55, precision=0.30250000000000005


100%|██████████| 25/25 [00:01<00:00, 13.83it/s]

Epoch 49: train_loss=0.6909905672073364, eval_loss=0.6897804737091064, accuracy=0.55, f1_score=0.3903225806451613, recall=0.55, precision=0.30250000000000005



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [36]:
# After training, output the predictions
# Convert predictions and true labels to a DataFrame for easy analysis
results_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': eval_preds
})

In [37]:
# Save the results to a CSV file
results_df.to_csv("/content/drive/MyDrive/model_predictions.csv", index=False)
print("Predictions saved to /content/drive/MyDrive/model_predictions.csv")

# Or print the first few rows of the results
print(results_df)

Predictions saved to /content/drive/MyDrive/model_predictions.csv
     true_label  predicted_label
0             1                0
1             0                0
2             0                0
3             1                0
4             1                0
..          ...              ...
195           0                0
196           0                0
197           0                0
198           1                0
199           1                0

[200 rows x 2 columns]


In [38]:
accuracy = accuracy_score(true_labels, eval_preds)
f1 = f1_score(true_labels, eval_preds, average='weighted')
recall = recall_score(true_labels, eval_preds, average='weighted')
precision = precision_score(true_labels, eval_preds, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.55
F1 Score: 0.3903225806451613
Recall: 0.55
Precision: 0.30250000000000005


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
